# Public charging points in Dutch municipalities: need or affluence?

**TIL6022 Python Programming – Group 15**

Members: Merel Vos, Mirte Batema, Famke Lemstra, Luca Preijde, Mats de Vries

Repository: [github.com/mirtebatema/group-15](https://github.com/mirtebatema/group-15)

In [1]:
# Setup: the tables in this report are read from the results of notebooks 6 to 8
from decimal import Decimal, ROUND_HALF_UP
import pandas as pd

PROCESSED = "data/processed/"


def stars(p):
    # Significance stars: * p < 0.05, ** p < 0.01, *** p < 0.001
    if p < 0.001:
        return "***"
    elif p < 0.01:
        return "**"
    elif p < 0.05:
        return "*"
    return ""


def signed(x, decimals=2):
    # Number with a + or - sign, rounded half up (0.305 becomes +0.31)
    rounded = Decimal(str(x)).quantize(Decimal(1).scaleb(-decimals), rounding=ROUND_HALF_UP)
    return f"{rounded:+}"


def p_text(p):
    # p-value as text, with '< 0.001' for very small values
    return "< 0.001" if p < 0.001 else f"{p:.3f}"

## 1. Introduction and motivation

The Netherlands aims for a fast transition to electric passenger cars. To support this, The National Charging Infrastructure Agenda (NAL), part of the national Climate Agreement, sets the ambition of adding just over 300,000 public and 500,000 semi-public charging points by 2030 (Binnenlands Bestuur, 2024; VNG, 2026). Public charging points are essential for a large part of the population. Only 27% of Dutch households can park on their own property, compared to 67% of current EV drivers (RVO et al., 2025).

Municipalities play a key role in the development of this network. They manage public space and are responsible for planning, permits and the direction of the rollout (VNG, 2026). However, substantial differences exist between municipalities. In the data used in this study, the density of regular public charging points ranges from 0.5 to 42.9 per 1,000 households. International research shows that higher-income areas often have better access to public charging (Chen et al., 2025). Whether similar patterns exist across Dutch municipalities remains an important question.

The central question is therefore whether the distribution of public charging infrastructure reflects local need, such as high population density and limited access to private parking, or socioeconomic characteristics, such as income and education levels. Using open data for all 342 Dutch municipalities, this study examines how demographic, socioeconomic and political characteristics are associated with public charging point density. Understanding these relationships can help assess whether the rollout reflects local needs or whether certain municipalities risk falling behind in the transition to electric mobility.

## 2. Research objective and questions

**Objectives**

1. Map the variation in public charging point density across Dutch municipalities.
2. Examine how socioeconomic, demographic, and political characteristics relate to that density.
3. Determine with a multivariate regression which characteristics explain the most variation when taken together.
4. Assess whether the rollout is linked mainly to need (population density, share of apartments) or to socioeconomic characteristics (income, education).

**Main research question**

To what extent do socioeconomic, demographic and political characteristics explain differences in the number of public charging points per 1,000 households between Dutch municipalities, and which characteristics have the greatest explanatory power?

*Table 1. Sub-questions and expected relations.*

| Sub-question | Expected relation (hypothesis) |
| --- | --- |
| SQ1. How does charging point density vary between municipalities? | Large differences between municipalities. |
| SQ2. How is charging point density related to income and education level? | Positive. More EV owners in richer municipalities. |
| SQ3. How is charging point density related to age structure? | Positive for a young population, negative for a large 65+ share. |
| SQ4. How is charging point density related to political preference? | Positive for a higher share of left-wing or climate-oriented votes. |
| SQ5. How is charging point density related to population density and housing type? | Positive. Dense municipalities have fewer private driveways, so more need for public charging. |

## 3. Contribution statement

As a group we defined the research question, the variables and the scope of the study. Everyone contributed to the code and the main responsibilities per person are described below.

**Mirte Batema**
- Coding: cleaning the raw datasets and preparing them for the analysis (notebooks 0), map of income (notebook 1.2) and tables for the main report
- Reporting: Analysis and results (sections 6.4–6.5) and AI statement

**Merel Vos**
- Coding: maps of charging point density, population density, age and education (notebooks 1, 2, 2.2, 3 and 5), boxplot and scatter plots (notebook 6.2) and robustness checks of the regression (notebook 8)
- Reporting: Data used (chapter 4) and Data pipeline (chapter 5)

**Famke Lemstra**
- Coding: map of left-wing vote share (notebook 4), README file
- Reporting: Contribution statement (chapter 3) and Analysis and results (sections 6.1–6.3)

**Luca Preijde**
- Coding: descriptive statistics and comparison of all factors (notebook 6, section 6.6); multivariate regression and model diagnostics (notebook 7)
- Reporting: Introduction (chapter 1), Research objective and questions (chapter 2), Conclusion (chapter 8) and Discussion (chapter 9)

**Mats de Vries**
- Coding: multivariate regression and model diagnostics (notebook 7), conversion of the report into the final notebook (main report)
- Reporting: Regression model (chapter 7) and References (chapter 10)

## 4. Data used

The unit of analysis is the municipality (342 municipalities). The study is cross-sectional: each variable comes from a single observation period, although these periods differ between 2024 and 2026 (see Table 2). The dependent variable is the number of regular public charging points per 1,000 households. Households are used instead of inhabitants because charging demand is more closely related to households and their car ownership than to the number of individual residents. This also makes comparisons between municipalities with different average household sizes more accurate. Table 2 provides an overview of the datasets (the raw data) that are used in data preparation before the analyses could be carried out.

*Table 2. Overview of the raw datasets in data/raw/.*

| File in data/raw/ | Content | Period | Source |
| --- | --- | --- | --- |
| laadpunten_per_gemeente.csv | Regular public charging points per municipality | July 2026 | NAL (2026) |
| inkomens_per_gemeente.csv | Private households and mean and median standardised income per municipality | 2024 (provisional) | CBS (2025a) |
| jongeren_per_gemeente_2026.csv, ouderen_per_gemeente_2026.csv | Share of residents aged 0–25 and 65+ per municipality | 2026 | CBS (2026) |
| tk2025_uitslag.csv | Results of the Tweede Kamer election of 29 October 2025 per municipality and party | 2025 | Kiesraad (2025) |
| kwb2024.zip | Full download of CBS Kerncijfers wijken en buurten 2024, with all districts and neighbourhoods | 2024 | CBS (2025b) |
| gemeenten_2024.gpkg | Municipal boundaries (layer gemeente_gegeneraliseerd), retrieved from the PDOK web service | 2024 | CBS (2024) |


The datasets are joined on municipality name. The sources write some names differently. CBS writes *'s-Gravenhage (gemeente)* instead of *Den Haag* and *Beek (L.)* instead of *Beek*, and the Kiesraad writes *Hengelo (O)* and *Bergen (NH)* without the final full stop. These names are mapped by hand to the spelling of the charging point file before merging: nine names in the CBS files and four in the election results. After this step all 342 municipalities match in every dataset. For the maps of education level and population density, the data is joined on the CBS municipality code (GMxxxx) instead, because both files contain this code. Joining all datasets on this code would be more robust, because it avoids spelling differences and makes municipal mergers visible.

Table 3 gives the definitions of all variables used in the analyses.

*Table 3. Definitions of the variables used in the analyses.*

| Variable | Definition | Unit |
| --- | --- | --- |
| **Charging point density** | Regular public charging points divided by the number of private households × 1,000 | charging points per 1,000 households |
| **Income** | Standardised household income, corrected for differences in household size. Both the mean and the median are reported; the median is less affected by a few very rich households | € 1,000 |
| **Age structure** | Share of residents aged 0–25 and share aged 65 and over, as a percentage of the total population | % |
| **Highly educated** | Persons aged 15–75 with hbo or wo as highest completed level, divided by all persons aged 15–75 with a known education level | % |
| **Left-wing vote share** | Votes for GroenLinks-PvdA, SP, Partij voor de Dieren, DENK, BIJ1, D66 and Volt, divided by all valid votes for party lists. A narrow definition without D66 and Volt is used as a robustness check. Postal votes from abroad are left out, because they do not belong to a municipality | % |
| **Population density** | Inhabitants per km² of land. Strongly right-skewed (from 23 on Schiermonnikoog to 6,868 in Den Haag, median 479), so its natural logarithm is used in the regression | inhabitants per km² |
| **Share of apartments** | Apartments as a percentage of all dwellings; a second indicator of the need for public charging | % |

## 5. Data pipeline

The data analysis consists of multiple notebooks. Each notebook reads its input from the raw data folder or from the processed data folder, and stores its output in figures or again in the processed data folder. The notebooks can be split into three main functions: data processing (0.1 to 0.6), exploration (1 to 5) and analysis (6 to 8).

1. **Collection.** Six datasets were downloaded and are stored in data/raw/, see Table 2.
2. **Extraction.** Notebook 0.5 uses the kwb2024 zip, extracts only the variables on education, population density and share of apartments, and saves them as kwb2024_per_gemeente.csv. It is stored in data/raw/ because it is input for several other notebooks.
3. **Data processing** (notebooks 0.1 to 0.6, and notebook 1). Each notebook prepares one variable from the raw data. Text values are converted to numbers, commas are replaced by points, total rows are removed, and municipality names are aligned with their spelling in the charging point file. Each notebook saves the processed data file for its variable in data/processed/. Notebook 1 combines the number of charging points, the number of households and the average income per municipality.
4. **Exploration per variable** (notebooks 1.2, 2, 2.2, 3, 4 and 5). These notebooks only read processed files and make a map per municipality; they do not save new data. Notebook 2 calculates the dependent variable, charging points per 1,000 households, from chargingpoints_households_income.csv. Notebook 1.2 maps income from the file of notebook 0.6, and notebooks 2.2, 3, 4 and 5 map population density, age structure, left-wing vote share and education level from the files of notebooks 0.4, 0.1, 0.3 and 0.2. All exploration notebooks use chargingpoints_households_income.csv to check that the municipality names match.
5. **Merging** (notebook 6). Notebook 6 joins all variables into one table with one row per municipality (all_variables_per_municipality.csv). It reads the processed files of notebooks 0.1–0.4 for age structure, left-wing vote share, education level and population density. Charging points, households and income are read from the raw files, and the share of apartments from the CBS extract, because no processed file contains these variables in the required form. This table is the input for all further analyses.
6. **Descriptive analysis** (notebooks 6 and 6.2). Notebook 6 uses the merged table directly, and notebook 6.2 reads it from all_variables_per_municipality.csv. Notebook 6 saves the correlation of each factor with charging point density in table_comparison_factors.csv, and the histogram and correlation matrix in figures/. Notebook 6.2 saves the boxplot and scatter plots in figures/. These outputs are not used by other notebooks.
7. **Regression** (notebook 7). Notebook 7 reads all_variables_per_municipality.csv. Inside the notebook, it gives the variables short names, takes the logarithm of population density, and converts all variables to z-scores. The regression results are saved in regression_results.csv, and the coefficient and residual plots in figures/.
8. **Stepwise regression** (notebook 8). Notebook 8 reads the same table, all_variables_per_municipality.csv, and prepares the variables in the same way as notebook 7. Its results are saved in three files in data/processed/ (stepwise_results.csv, all_subsets_results.csv and composite_model_results.csv), and two figures are saved in figures/.

## 6. Analysis and results

This chapter answers the sub-questions one by one, and section 6.6 compares all factors side by side. All factors are linked to charging point density, but each explains only part of the variation on its own: R² ranges from 0.03 for mean income to 0.23 for population density on a logarithmic scale.

### 6.1 Variation in charging point density (SQ1)

This section examines how much public charging point density differs between municipalities. For each municipality, the number of regular public charging points was divided by the number of private households and multiplied by 1,000. Figure 1 shows the result on a map (notebook 2). Figure 2 and Figure 3 show the distribution as a histogram (notebook 6) and a boxplot (notebook 6.2).

![Map of charging point density](figures/map_chargingpoints_per_1000households.png)

*Figure 1. Regular public charging points per 1,000 households per municipality, July 2026.*

![Histogram of charging point density](figures/figure_distribution_of_public_charging_point_density.png)

*Figure 2. Distribution of charging points per 1,000 households; dashed line: median (notebook 6).*

![Boxplot of charging point density](figures/figure_boxplot_charging_point_density.png)

*Figure 3. Boxplot of charging points per 1,000 households (notebook 6.2). Points above the upper whisker are outliers (more than 1.5 × the interquartile range above the third quartile).*

The map shows a clear regional pattern. Municipalities in the Randstad and parts of Noord-Brabant are among the darkest, as is Sluis in the far south-west. Most of the north and east of the country is much lighter. On average, a municipality has 15.0 public charging points per 1,000 households (standard deviation 6.8), and half of all municipalities lie between 9.9 and 19.3.

The spread between the extremes is large. Loon op Zand has the highest density (42.9), followed by Katwijk (33.5), Culemborg (33.1), Barendrecht (32.9) and Leiderdorp (32.5). At the other end are Renkum (0.5), Berkelland (1.4), Ameland (2.2), Dantumadiel (2.5) and Pekela (3.2).

The histogram shows that the distribution is right-skewed. Most municipalities have between 5 and 25 charging points per 1,000 households, but a small group has far more. As a result, the mean (15.0) is higher than the median (13.5).

In the boxplot, only Loon op Zand and Katwijk lie above the upper whisker. Loon op Zand stands out in particular, with almost ten charging points more than the next municipality. On the lower side there are no outliers: the whisker extends down to the lowest value of about 0.5. According to the data, this value belongs to Renkum.

**Answer to SQ1.** Public charging point density differs strongly between municipalities, by a factor of more than 80 between the lowest and the highest. Most municipalities lie close together, and a few, Loon op Zand in particular, are far above the rest.

### 6.2 Income and education (SQ2)

This section examines whether municipalities with a higher income or a higher education level have more public charging points. Two income measures from CBS (2024) are used: the mean and the median standardised household income. Education is measured as the share of residents aged 15 to 75 with HBO or WO as their highest completed level (CBS Kerncijfers wijken en buurten, 2024). Each variable is correlated with charging point density using the Pearson correlation (notebook 6) and plotted against it in a scatter plot (Figure 9, notebook 6.2). Income and education level are also shown on a map (Figure 4, notebook 1.2, and Figure 5, notebook 5).

**Income**

![Map of mean income per municipality](figures/map_income_per_municipality.png)

*Figure 4. Mean standardised income per municipality, 2024 (notebook 1.2).*

The two income measures give clearly different results. With mean income, the correlation is weak (r = 0.17, p = 0.001, N = 336). With median income, it is twice as strong (r = 0.34, p < 0.001, N = 339). The number of municipalities is slightly lower than 342, because CBS does not publish income figures for a few very small municipalities.

The scatter plot of mean income in Figure 9 explains this difference. Most municipalities lie close together between roughly €35,000 and €50,000, but a few municipalities, such as Laren, Blaricum and Bloemendaal, lie far to the right, up to about €155,000. Their charging point density is average, which flattens the regression line. When the municipalities with a mean income above €100,000 are left out, the correlation rises to r = 0.27. The median is less sensitive to a few very rich households and is therefore used as the income measure in the rest of this study.

**Education**

![Map of education level per municipality](figures/map_education_per_municipality.png)

*Figure 5. Share of highly and low educated residents (aged 15–75) per municipality, 2024 (notebook 5).*

The map of highly educated residents shows a concentration in the Randstad, around Utrecht and in several university cities, while the share is lower in large parts of the north-east and Zeeland. The map of low educated residents shows roughly the mirror image. The share of highly educated residents ranges from 12.4% in Urk to 59.0% in Rozendaal, with a median of 28.7%.

The correlation with charging point density is positive and of about the same size as for median income (r = 0.31, p < 0.001, N = 342; Spearman ρ = 0.34; R² = 0.093). This is not surprising, as education and median income are themselves moderately related (r = 0.54, Figure 10). Both variables therefore partly reflect the same socioeconomic position of a municipality. Whether each still has an effect when the other is taken into account is tested in chapter 7.

**Answer to SQ2.** Charging point density rises with both income and education. Median income (r = 0.34) is slightly more strongly linked than the share of highly educated residents (r = 0.31), and each explains about 10% of the variation between municipalities.

### 6.3 Age structure (SQ3)

This section examines whether the age composition of a municipality is linked to charging point density. Two variables from CBS (2026) are used: the share of young residents (aged 0–25) and the share of older residents (aged 65 and over), both as a percentage of the total population. Figure 6 shows both shares on a map (notebook 3). For each group, the Pearson correlation with charging point density, a linear regression line and R² were calculated (notebook 3); the scatter plots are included in Figure 9.

![Map of age structure per municipality](figures/map_age_per_municipality.png)

*Figure 6. Share of residents under 25 and aged 65+ per municipality, 2026 (notebook 3).*

The two maps largely mirror each other. Young residents form a relatively large share in Flevoland, around Urk and in parts of the centre of the country. Older residents are overrepresented in more peripheral areas, such as Zeeland, the Wadden Islands and South Limburg.

*Table 4. Correlation and regression results for age structure.*

| Age group | Pearson r | p-value | R² | Slope |
| --- | --- | --- | --- | --- |
| Share under 25 (%) | +0.29 | < 0.001 | 0.085 | +0.69 points per percentage point |
| Share 65+ (%) | −0.34 | < 0.001 | 0.117 | −0.63 points per percentage point |

Municipalities with a younger population have more public charging points, and municipalities with older residents have fewer. Each additional percentage point of residents aged 65 and older corresponds to 0.63 fewer charging points per 1,000 households. The relation with the share of older residents is slightly stronger than with the share of young residents.

The two age variables are strongly negatively related (r = −0.82, Figure 10), so they largely describe the same contrast between younger and older municipalities. The correlation matrix also shows that municipalities with many older residents tend to have a lower population density (r = −0.45) and fewer apartments (r = −0.42). Part of the age effect may therefore reflect differences in urbanity rather than age itself. Section 6.5 discusses population density, and chapter 7 tests whether age still has an effect when density is taken into account.

**Answer to SQ3.** Age structure is linked to charging point density in the expected direction: positive for the share of young residents and negative for the share of older residents. The share of residents aged 65 and older explains slightly more of the variation (11.7%) than the share of young residents (8.5%).

### 6.4 Political preference (SQ4)

This section examines whether municipalities with a higher share of left-wing voters have a higher density of public EV charging points. The results of the 2025 Dutch parliamentary elections (Tweede Kamer) were used to calculate the percentage of left-wing votes in each municipality. Two definitions of left-wing parties were considered. The broad definition includes GroenLinks-PvdA, SP, Partij voor de Dieren, DENK, BIJ1, D66, and Volt, while the narrow definition excludes D66 and Volt (Figure 7, notebook 4). Both definitions were used to investigate whether there is a correlation between the percentage of left-wing votes and the number of public EV charging points per 1,000 households (Table 5).

![Map of left-wing vote share per municipality](figures/map_left_wing_votes_per_municipality.png)

*Figure 7. Left-wing vote share per municipality, Tweede Kamer election 2025 (notebook 4).*

There is no major difference in the geographical patterns shown by the two maps. However, the map using the broad definition is generally darker because more parties are included, resulting in a higher percentage of left-wing votes. Higher percentages are particularly visible in the Randstad region and larger cities. Amsterdam has one of the highest shares of left-wing voters, at 71.2%, while Urk has one of the lowest, at only 1.9%.

*Table 5. Correlation results for left-wing vote share.*

| Left-wing vote | Pearson r | p-value | R² |
| --- | --- | --- | --- |
| Broad definition | +0.22 | < 0.001 | 0.048 |
| Narrow definition | +0.15 | 0.004 | 0.024 |

Table 5 shows the correlation between the percentage of left-wing votes and the density of public EV charging points per municipality. Municipalities with a higher share of left-wing voters tend to have slightly more public charging points, although the correlation is weak. The correlation is even weaker when using the narrow definition of left-wing parties, suggesting that D66 and Volt voters contribute to this relationship. Furthermore, the share of left-wing votes is barely correlated with median income (r = 0.06), indicating that political preference measures something different from municipal wealth.

**Answer to SQ4.** Political preference has a weak positive correlation with public EV charging point density (r = 0.22). This means that political preference explains approximately 5% of the variation in charging point density between municipalities.

### 6.5 Population density and housing type (SQ5)

This section examines whether municipalities with a higher population density have more public EV charging points, as these areas may have a greater need for public charging infrastructure. To investigate this relationship, we used population density (inhabitants per km²) (Figure 8, notebook 2.2) and the percentage of apartments in the total housing stock. Both variables were obtained from the CBS Kerncijfers Wijken en Buurten 2024 dataset and compared with public EV charging point density (notebook 6). Since population density is strongly skewed, we also calculated the Spearman correlation and the correlation using the logarithm of population density to better understand the relationship.

![Map of population density per municipality](figures/map_population_density_per_municipality.png)

*Figure 8. Population density per municipality in inhabitants per km² land, logarithmic colour scale, 2024 (notebook 2.2).*

Population density is particularly high in the Randstad region, while municipalities in the north of the Netherlands generally have lower population densities. It ranges from 23 inhabitants per km² in Schiermonnikoog to 6,868 in Den Haag, with a median of 479.

*Table 6. Correlation of population density and share of apartments with charging points per 1,000 households. All p < 0.001.*

| Variable | N | Pearson r | Spearman ρ | R² |
| --- | --- | --- | --- | --- |
| Population density | 342 | +0.41 | +0.51 | 0.169 |
| log(population density) | 342 | +0.48 | – | 0.232 |
| Apartments (% of dwellings) | 342 | +0.34 | +0.37 | 0.117 |

Table 6 shows a positive correlation between population density and the number of public EV charging points per 1,000 households. Municipalities with a higher population density generally have more public charging points. The correlation is stronger when using the logarithm of population density (r = 0.48), which is associated with approximately 23% of the variation in charging point density.

The share of apartments shows a similar relationship. This supports the expectation that public charging infrastructure is more common in areas where residents have fewer opportunities to charge their vehicles at home. However, population density and the share of apartments are strongly correlated (r = 0.79). Therefore, we only include one of these variables in the regression, because they measure similar characteristics of municipalities.

**Answer to SQ5.** Denser municipalities with more apartments have clearly more public charging points (r = 0.41 for density, 0.48 for log density). Of all single factors, this indicator of need shows the strongest link.

### 6.6 Comparison of all factors

This section compares all the factors examined in the previous sections to determine which are most strongly related to the number of public EV charging points per 1,000 households. All variables are combined into one dataset per municipality (notebook 6). The Pearson and Spearman correlations are calculated to compare the strength of the relationships between each factor and charging point density (Table 7 and Figure 9). In addition, a correlation matrix is used to examine how the different factors are related to each other (Figure 10). This comparison helps identify which variables are most relevant to include in the regression analysis in the next chapter.

*Table 7. Correlation of each explanatory variable with charging points per 1,000 households, sorted by strength. All correlations shown are significant at p < 0.01.*

| Variable | Group | N | Pearson r | Spearman ρ | R² |
| --- | --- | --- | --- | --- | --- |
| Population density (inh./km²) | Need | 342 | +0.41 | +0.51 | 0.169 |
| Apartments (% of dwellings) | Need | 342 | +0.34 | +0.37 | 0.117 |
| Residents 65+ (%) | Demographic | 342 | −0.34 | −0.38 | 0.117 |
| Median standardised income (k€) | Socioeconomic | 339 | +0.34 | +0.34 | 0.113 |
| Highly educated (%) | Socioeconomic | 342 | +0.31 | +0.34 | 0.093 |
| Residents under 25 (%) | Demographic | 342 | +0.29 | +0.37 | 0.085 |
| Left-wing vote share (%) | Political | 342 | +0.22 | +0.24 | 0.048 |
| Mean standardised income (k€) | Socioeconomic | 336 | +0.17 | +0.34 | 0.030 |

![Scatter plots of all factors](figures/figure_scatter_all_factors.png)

*Figure 9. Charging points per 1,000 households against each explanatory variable, with least-squares line (notebook 6.2). Population density is shown on a logarithmic axis.*

![Correlation matrix of all variables](figures/figure_correlationmatrix.png)

*Figure 10. Correlation matrix of all variables (notebook 6).*

Table 7 shows that population density has the strongest positive correlation with public EV charging point density (r = 0.41). This is followed by the share of apartments, the percentage of residents aged 65 and older, and median income, all with correlations of approximately 0.34. Education shows a slightly weaker relationship, while the percentage of left-wing votes has the weakest correlation. This suggests that factors related to the need for public charging infrastructure are more strongly associated with charging point density than socioeconomic factors.

The correlation matrix (Figure 10) also shows that some variables are strongly related to each other. For example, the percentage of residents under 25 and over 65 has a strong negative correlation (r = −0.82), while population density and the share of apartments have a strong positive correlation (r = 0.79). Left-wing vote share and education are also strongly correlated (r = 0.78), as are mean and median income (r = 0.66). Because these variables are closely related, including both variables from each pair in the regression could make it difficult to identify their individual effects. Therefore, we will consider these relationships when selecting variables for the regression analysis in the next chapter.